## 5. 파일럿 — 10쌍 P1a 기술 확인 (본실험 전 필수)

본실험 전 **10쌍·1 seed로 P1a만** 돌려 (a) 세션 폐기율, (b) 기저 gap 부호,
(c) L25 조건이 실제로 움직이는지 확인한다. P2·전체 sweep은 그다음.


## 0. GPU 확인
3B fp16은 T4(16GB)에 올라간다. GPU 없으면 런타임 유형을 T4로.


In [ ]:
!nvidia-smi -L


## 1. repo clone (main)
코드는 **main**에서. 이미 있으면 main 최신으로 정렬.


In [ ]:
REPO_URL = "https://github.com/deanjs/instruction-adherence.git"
BRANCH = "main"
import os
if not os.path.isdir("instruction-adherence"):
    !git clone --branch {BRANCH} {REPO_URL}
%cd instruction-adherence
!git checkout {BRANCH} && git pull origin {BRANCH}
!git log --oneline -1


## 2. 의존성
`AttentionInterface` 등록에 transformers>=4.51. Colab torch(CUDA)는 유지.


In [ ]:
!pip install -q "transformers>=4.51.0" "accelerate>=0.26.0"
import torch, transformers
print("transformers", transformers.__version__, "| cuda", torch.cuda.is_available())


## 3. 검증 (게이트) — 개입 하네스 불변식

**PASS여야 실측으로 넘어간다.** V1 표준경로=SDPA · V2 no-op 불변 · V3 지침 patch≈0 ·
V4 GQA 단위(P1a=KV group·P2=query head) · V5 P2 질량 보존 · V6 λ=1 항등 ·
V7 L25 이식이 점수를 움직이는지(sanity). 크기 무관 → 1.5B fp32.


In [ ]:
!python src/stage3_intervention.py --validate


## 4. A분할 — 기저 gap (준수 − 손상)

개입 없이 손상/준수 baseline만. gap = 완전 회복 목표 = Recovery Ratio 분모.
Drive에 append(재개 가능).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
OUT = "/content/drive/MyDrive/instruction-adherence/stage3_intervention.jsonl"
os.makedirs(os.path.dirname(OUT), exist_ok=True)
!python src/stage3_intervention.py --run-a --n-seeds 3 --out "{OUT}"


## 5. 파일럿 — 10쌍 P1a 기술 확인 (본실험 전 필수)

본실험 전 **10쌍·1 seed로 P1a만** 돌려 (a) 세션 폐기율, (b) 기저 gap 부호,
(c) L25 조건이 실제로 움직이는지 확인한다. P2·전체 sweep은 그다음.


In [ ]:
!python src/stage3_intervention.py --run-b --p1a \
    --max-pairs 10 --n-seeds 1 --out "{OUT}"


## 6. 본실험 B — 개입 실행 (P1a 주 · P2 보조)

- **P1a 주 조건 = L25 단독 × 전 KV group**(코드 K/V 이식). 전 층 각각을 같은 규모로 돌려
  L25 아닌 층들이 **규모-일치 귀무분포**가 된다. `p1a_full`은 회복 상한.
- **음성 대조**(L25 규모): no-op·지침·무관 코드·코드 외 → Δ≈0 기대.
- **P2**: 지침 α×λ 곡선(단조성) + 전 층 국소.

기본 세트는 pair당 ~183 config. `--sweep`을 붙이면 group별·층×head 순회까지(수백 config↑).
P1a만 빠르게 보려면 `--p1a`만.


In [ ]:
!python src/stage3_intervention.py --run-b --p1a --p2 --n-seeds 3 --out "{OUT}"


**(선택) 국소화 스윕** — group별·후보 층 query head 28개 순회.


In [ ]:
!python src/stage3_intervention.py --run-b --p1a --p2 --sweep --n-seeds 3 --out "{OUT}"


## 7. 집계 — cluster bootstrap · 층 귀무 · Recovery Ratio · λ 추세

이름쌍·seed cluster bootstrap 95% CI, L25가 나머지 층(같은 규모) 귀무분포 상위 꼬리인지,
Recovery Ratio CI(분자·분모 독립 부트), P2 λ 단조 추세(Spearman).


In [ ]:
!python src/stage3_intervention.py --summary-only --out "{OUT}"


## 8. 결과 내려받기 (선택)
`stage3_intervention.jsonl`은 사전 등록 기록.


In [ ]:
from google.colab import files
files.download(OUT)
